In [70]:
!uv pip install fastapi uvicorn python-multipart numpy pillow --quiet

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.testclient import TestClient

from io import BytesIO
from PIL import Image
import numpy as np

import tensorflow as tf

In [ ]:
MODEL = tf.keras.models.load_model("./models/plant_village_v1.keras")
CLASS_NAMES = ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']

app = FastAPI()
client = TestClient(app)

In [66]:
def get_request(endpoint):
    response = client.get(endpoint)
    print(response.status_code)
    print(response.json())

def post_request(endpoint, body):
    response = client.post(endpoint, data=body)
    print(response.status_code)
    print(response.json())

In [40]:
@app.get("/ping")
async def ping():
    return 'Hello, I am alive'

get_request("/ping")

200
Hello, I am alive


In [58]:
def read_file_as_image(data) -> np.ndarray:
    image = np.array(Image.open(BytesIO(data)))
    return image

In [ ]:
@app.post("/predict")
async def predict(
    file: UploadFile = File(...)
):
    image = read_file_as_image(await file.read())
    np.expand_dims(image, 0)
    MODEL.predict(image)
    print(image)

post_request("/predict", '')

400
{'detail': 'There was an error parsing the body'}


In [69]:
get_request("/doc")

404
{'detail': 'Not Found'}
